# Week 4 & Final Integration: DistilBERT & Aspect-Based Sentiment Analysis (ABSA)
### Project: Intelligent Customer Review Sentiment Analysis
This notebook demonstrates:
1. **Model 3: DistilBERT Transformer** Architecture & Tokenization
2. Fine-tuning for 3-Class Sentiment Classification
3. **3-Model Benchmark Comparison** (LSTM vs BiLSTM vs DistilBERT)
4. **Advanced Module**: Aspect-Based Sentiment Analysis (ABSA)

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

print('Hugging Face Transformers & PyTorch ready!')

## 1. DistilBERT Tokenization & Architecture
DistilBERT retains 97% of BERT's language understanding while being 40% smaller and 60% faster.

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
sample_review = 'The camera quality is excellent, but the battery backup is terrible.'
encoded = tokenizer(sample_review, padding='max_length', max_length=64, truncation=True, return_tensors='pt')

print('Input IDs:', encoded['input_ids'][0][:15])
print('Attention Mask:', encoded['attention_mask'][0][:15])

## 2. Benchmark Comparison: LSTM vs BiLSTM vs DistilBERT
Evaluated on identical test partition (2,250 reviews across 3 classes).

In [ ]:
with open('../backend/data/benchmark_results.json') as f:
    benchmarks = json.load(f)

bench_df = pd.DataFrame(benchmarks['models'])[['id', 'name', 'accuracy', 'f1_score', 'avg_latency_ms', 'model_size_mb']]
display(bench_df)

# Plot comparisons
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=bench_df, x='name', y='accuracy', ax=axes[0], palette='crest')
axes[0].set_title('Model Accuracy Comparison (%)')
axes[0].set_ylim(70, 100)
axes[0].set_ylabel('Accuracy (%)')

sns.barplot(data=bench_df, x='name', y='avg_latency_ms', ax=axes[1], palette='rocket')
axes[1].set_title('Inference Latency per Review (ms)')
axes[1].set_ylabel('Milliseconds (ms)')

plt.tight_layout()
plt.show()

## 3. Advanced Feature: Aspect-Based Sentiment Analysis (ABSA)

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../backend'))
from app.models.absa_engine import AspectBasedSentimentAnalyzer

absa = AspectBasedSentimentAnalyzer()
test_cases = [
    'The camera quality is excellent, but the battery backup is terrible.',
    'Screen is bright and responsive, sound is crystal clear, however customer support refused warranty.',
    'Overpriced laptop with mediocre keyboard, but the performance is snappy.'
]

for t in test_cases:
    res = absa.analyze(t)
    print('='*70)
    print(f'REVIEW:  {t}')
    print(f'OVERALL: {res["overall_aspect_sentiment"]}')
    print('ASPECTS:')
    for a in res['detected_aspects']:
        print(f'  • {a["aspect"]}: {a["sentiment"]} (keyword: {a["keyword"]})')
    print('\n')